# EMG Data Pre-processing Script 

This script loops through the raw EMG files, preprocesses the raw data, and extracts variance, WL, and RMS using a sliding window. The preprocessed data can be saved to an Excel file, and the results can be visualized with a GUI.

## Library Importations 

In [ ]:
import os.path as op
import mne 
import os
import numpy as np 
import pandas as pd
from sklearn.model_selection import train_test_split, KFold
from matplotlib import pyplot as plt
import pickle

In [ ]:
# sanity check for columns to remove 
trigger_info = pd.read_csv("trigger_counts.csv")
print(trigger_info.loc[trigger_info["Trigger_number"] != 60, ["Subject", "Nap"]])

## Defining initial variables 

In [ ]:
to_keep=['Fp1', 'Fp2', 'C3', 'Cz', 'C4', 'P3', 'Pz', 'P4', 'O1', 'O2','EOG1','EOG2','Corr', 'Zygo', 'Menton','Trigger']
eeg_ch= ['Fp1', 'Fp2', 'C3', 'Cz', 'C4', 'P3', 'Pz', 'P4', 'O1', 'O2']
emg_ch= ['Corr', 'Zygo', 'Menton'] # Menton is chin EMG for sleep scoring
eog_ch= ['EOG1', 'EOG2']
trigger_ch= ['Trigger']
subject_exclude = ["NL01SS", "NL02IF",
                   "NL05WW01", "RL12JL03", "RL07BR02", "RL22AC05",
                   "RL16CM", "RL11JH","RL18CL", "RL21AC"]  # trials to exclude based on trigger count 


inter_trigger_length = 30
num_epochs = 60 # 60 epochs for each session 

raw_path= "/Users/zeynepozkaya/Desktop/SoundSleep/Python_Scripts/full_EEG_dataset"
current_index=0
inter_trigger_length=10
window = 50 
step = 1 

## Helper functions 

In [ ]:
# pre-processes data for each subject and returns df with session divided into 60 epochs w meta-data attached 
def pre_process_subjets(subject,block):
    global raw_path
    global frq

    subject_name = subject + block 
    file= op.join(raw_path,'{}.edf'.format(subject_name))
    raw =  mne.io.read_raw_edf(file,preload=True)

    if 'Fp1/F3' in raw.info['ch_names']:
        mne.rename_channels(info=raw.info,mapping={'Fp1/F3':'Fp1' ,'Fp2/F4':'Fp2'})
        
    if '36' in raw.info['ch_names']:
        mne.rename_channels(info=raw.info,mapping={'36':'Corr' ,'37':'Zygo','38':'Menton'})

    if 'E1' in raw.info['ch_names']:
        mne.rename_channels(info=raw.info,mapping={'E1':'EOG1' ,'E2':'EOG2'})
    
    if 'Corru' in raw.info['ch_names']:
        raw.rename_channels({'Corru': 'Corr'})


    for ch in raw.info['ch_names']:
        if ch not in to_keep:
            raw.drop_channels([ch]) # only keeps channel that contains the trigger (where a stimulus was presented )

    raw.set_channel_types(mapping={'Corr':'emg','Zygo':'emg','Menton':'emg','Trigger':'stim','EOG1':'eog','EOG2':'eog'})

    raw= raw.resample(sfreq=250)

    filter_params_emg = {'lpass': 100,'hpass': 10,'notches': [50]}
    raw.filter(l_freq=filter_params_emg['hpass'],h_freq=filter_params_emg['lpass'],picks=emg_ch)
    filter_params_eeg_eog = {'lpass': 70,'hpass': 0.3,'notches': [50]}
    raw.filter(l_freq=filter_params_eeg_eog['hpass'],h_freq=filter_params_eeg_eog['lpass'],picks=eeg_ch+eog_ch)
    raw.notch_filter(filter_params_eeg_eog['notches'], method='fft', picks=emg_ch+eeg_ch+eog_ch)

    frq=raw.info['sfreq']

    # create dataframe based on events 
    events_all= mne.find_events(raw) # Get all the events (triggers) in your EEG data
    events= mne.pick_events(events_all,include=[201,202]) # Pick events of interest (where a stimulus was presented)

    df_triggers=pd.DataFrame(data=events,columns=['Time(Sample)','dunno','Trigger'])
    df_triggers.drop(columns=['dunno'],inplace=True)
    df_triggers['Time(s)']=df_triggers['Time(Sample)']/raw.info['sfreq']


    # incorporating meta_data 
    cols_inc = ["Subject","Nap_ID","Trigger", "Expected_Muscle","Nb_Corr","Nb_Zygo","Is_Correct"] # columns with relevant information from dataframe 
    trial_info = pd.read_csv("Trial_information_narcolepsy.csv", usecols=cols_inc) 
    expected_muscle = ["Corr", "Zygo"]

 
    epochs = mne.Epochs(raw, events, tmin=-0, tmax=9,baseline=None, detrend=0,
                    reject=None, preload=True, on_missing='warn')

   


    epochs.metadata = trial_info[(trial_info["Subject"] == subject) 
                                & (trial_info["Trigger"].isin([201.0, 202.0]))
                                & (trial_info["Nap_ID"] == int(block))]

    # adding metadata column for true activation
    # add another column for neither muscle being activated  
    true_activations = [] 
    for i in range(len(epochs.metadata)):
        true_ind = expected_muscle.index(epochs.metadata.iloc[i]["Expected_Muscle"]) 
        if (epochs.metadata.iloc[i]["Is_Correct"] == 0):
            if(epochs.metadata.iloc[i]["Nb_Corr"] < 3 and epochs.metadata.iloc[i]["Nb_Zygo"] < 3):
                true_activations.append("None")
            else:
                true_activations.append(expected_muscle[true_ind-1])
        else:
            true_activations.append(expected_muscle[true_ind])


    epochs.metadata["True_activation"] = true_activations
    return epochs, df_triggers 

In [ ]:
# extracts features in a sliding window of size window samples with a step size of a certain number of samples 
def get_features(epoch):
     # pad epoch to preserve sample number 
     pad_left  = window // 2
     pad_right = window - 1 - pad_left   
     
     epoch_padded = epoch
     epoch_padded = np.pad(epoch, (pad_left, pad_right), mode="edge")  

     # take sliding window 
     epoch_sw = np.lib.stride_tricks.sliding_window_view(epoch_padded,window)[::step]

     var = np.var(epoch_sw, axis=-1) # calculate variance over window 
     rms =  np.sqrt((1/window)*np.sum(epoch_sw**2, axis=-1)) # calculate rms over window  
     wl = np.sum(np.abs(np.diff(epoch_sw, axis=1)), axis=1) # calculate wl over window  

  
     return var, rms, wl 


In [ ]:
# function for looping through all files, pre-processing, performing feature extraction, and creating data frame to be used for classification 
def make_features_df(subject_epoch,subject,block):
    features = pd.DataFrame(
        index=range(num_epochs),
        columns=[
            "Subject",
            "Nap Number",
            "Triggers_Order_Nap", # epochs 
            "True_Muscle_Activated",
            "Num_Contractions_Zygo",
            "Num_Contractions_Corr",
            "WL_Zygo", # three features being used 
            "Var_Zygo",
            "RMS_Zygo", 
            "MAVS_Zygo",
            "WL_Corr",
            "Var_Corr",
            "RMS_Corr", 
            "Zygo", # processed EMG signal for epoch 
            "Corr"
        ]
        )   
    
    for t in range(len(subject_epoch)): 
        # extract epoch  
        epoch_zygo = np.squeeze(subject_epoch[t].get_data(picks=['Zygo']))
        epoch_corr = np.squeeze(subject_epoch[t].get_data(picks=['Corr']))

        epoch_var_zygo, epoch_rms_zygo, epoch_wl_zygo = get_features(epoch_zygo)
        epoch_var_corr, epoch_rms_corr, epoch_wl_corr = get_features(epoch_corr)


        # fill dataframe 
        features.loc[t] = [
            subject, 
            int(block),
            t + 1,
            subject_epoch[t].metadata['True_activation'].iloc[0],
            subject_epoch.metadata.iloc[t]["Nb_Zygo"],
            subject_epoch.metadata.iloc[t]["Nb_Corr"],
            epoch_wl_zygo,
            epoch_var_zygo, 
            epoch_rms_zygo, 
            epoch_wl_corr,
            epoch_var_corr, 
            epoch_rms_corr, 
            epoch_zygo, 
            epoch_corr
        ]
    
    return features 

## Extracting features 

In [ ]:
i = 0 
features_all = []

# walk through directory to pre-processing and extract features
for root,dirs,files in os.walk(raw_path):
    for file in files:
        if "dpa" not in file and ".DS_Store" not in file: # so only take each subject once 
            subject = file.split(".")[0][:-2]
            block = file.split(".")[0][-2:]
            print(f"Processing{'.'*65}{subject}{block}")
            
            # skip over subjects with incorrect epcoh number 
            if (subject == "NL02IF" or 
                subject == "NL05WW" or 
                subject == "NL01SS" or 
                subject == "RL11JH" or 
                subject == "RL12JL" or 
                subject == "RL07BR"):
                continue
        
        subject_epoch, _ = pre_process_subjets(subject,block)
        features = make_features_df(subject_epoch,subject,block)
        features_all.append(features)
    

In [ ]:
features_all.to_excel("training_features.xlsx", index=False)

## GUI

In [ ]:
subject_plt = "NL03JV"
nap_plt = "02"
subject_df = features_all.loc[(features_all["Subject"] == subject_plt) & (features_all["Nap Number"] == int(nap_plt))] # extract subj info

In [ ]:
#GUI
current_index=0
inter_trigger_length=10
window = 50
step = 1

def plot_figure(t):    
    zygo_contractions = np.array(subject_df["Num_Contractions_Zygo"])[t]
    corr_contractions = np.array(subject_df["Num_Contractions_Corr"])[t]

    epoch_zygo = np.array(subject_df["Zygo"].tolist())

    # can also plot features 
    epoch_zygo_rms = np.array(subject_df["Zygo"].tolist())
    epoch_zygo_var = np.array(subject_df["Var_Zygo"].tolist())
    epoch_zygo_wl = np.array(subject_df["WL_Zygo"].tolist())

    epoch_corr = np.array(subject_df["Corr"].tolist())
    epoch_corr_rms = np.array(subject_df["RMS_Corr"].tolist())
    epoch_corr_var = np.array(subject_df["Var_Corr"].tolist())
    epoch_corr_wl = np.array(subject_df["WL_Corr"].tolist())

    fig, ax1 = plt.subplots(1,2, figsize=(20, 4))

    fig.suptitle(f"Epoch {t + 1}")

    # Corr subplot
    ax1[0].plot(epoch_corr[t], color="blue", label="Corr")
    ax1[0].set_ylim(-200, 200)
    ax1[0].set_title(f"Zygo: {zygo_contractions}, Corr: {corr_contractions}")
    ax1[0].set_ylabel("Zygo EMG [V]")
    ax1[0].set_xlabel("Samples")

    # Zygo subplot
    ax1[1].plot(epoch_zygo[t], label="Zygo",color="black")
    ax1[1].set_ylim(-200, 200)
    ax1[1].set_ylabel("Zygo EMG [V]")
    ax1[1].set_xlabel("Samples")
 
    # Connect mouse key press events
    fig.canvas.mpl_connect('key_press_event', on_key)

    plt.tight_layout()
    plt.suptitle(f"Subj. {subject_plt} | Nap {nap_plt} | Epoch {t}")
    plt.show()


# Keyboard press event handler
def on_key(event):
    global current_index, fig, features_all
    key = event.key
        
    if event.key == 'right':  # Move to next figure
        current_index = (current_index + 1) % len(subject_df)  # Loop to the start
        plt.close()  # Close current figure
        plot_figure(current_index)  # Plot the next figure
    elif event.key == 'left':  # Move to previous figure
        current_index = (current_index - 1) % len(subject_df)  # Loop to the end
        plt.close()  # Close current figure
        plot_figure(current_index)  # Plot the previous figure

    elif event.key == 'escape':  
        print("Quitting the plot!")
        plt.close(fig)  
        
# Plot the first figure
plot_figure(38)
